# 第23章　カスタムDatasetを書く ― 医療画像の読み込みを自作する**『医療診断支援AIを自分で作る（基礎編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-basic

## 23.1　対応表から読むDataset

In [ ]:
import torch, pydicom, numpy as npfrom torch.utils.data import Datasetclass CTDataset(Dataset):    def __init__(self, df, transform=None):        self.df = df.reset_index(drop=True)     # 対応表（case_id, image_path, label）        self.transform = transform    def __len__(self):        return len(self.df)    def __getitem__(self, idx):        row = self.df.iloc[idx]        ds = pydicom.dcmread(row["image_path"])          # DICOMを読む        hu = ds.pixel_array * ds.RescaleSlope + ds.RescaleIntercept  # HU値へ        img = ct_window(hu, center=40, width=400)        # 前処理（レシピ集の章）        img = torch.from_numpy(img).float().unsqueeze(0) # (1, H, W)        label = int(row["label"])        if self.transform:            img = self.transform(img)        return img, label

## 23.2　3次元（NIfTI）のDataset

In [ ]:
# 読み込みもMONAIのCompose（LoadImaged）が担当するので、ここではnibabelを直接呼ばないclass VolumeDataset(Dataset):    def __init__(self, df, transform=None):        self.df, self.transform = df.reset_index(drop=True), transform    def __getitem__(self, idx):        row = self.df.iloc[idx]        data = {"image": row["image_path"], "label": row["label_path"]}        if self.transform:            data = self.transform(data)          # MONAIのCompose（読み込み含む）        return data["image"], data["label"]    def __len__(self):        return len(self.df)# この例の transform は「単一の辞書を返す」ものに限る。第15章の RandCropByPosNegLabeld（num_samples>1）の# ように辞書のリストを返す変換を使うなら、リストのまま返して MONAI の DataLoader（list_data_collate）で束ねる

## ミニ・トラブルシュート ― DataLoaderのデッドロックと、結果が毎回変わる問題

In [ ]:
def seed_everything(seed=42):    import random, os    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)    torch.cuda.manual_seed_all(seed)    torch.backends.cudnn.deterministic = True    # 畳み込みを決定的アルゴリズムに    torch.backends.cudnn.benchmark = False        # 速度優先の非決定的最適化を切るdef worker_init_fn(worker_id):                     # 各ワーカーにも異なるが再現可能なシードを    np.random.seed((torch.initial_seed() + worker_id) % 2**32)   # 先に足してから丸める（上限超え防止）loader = DataLoader(train_ds, batch_size=8, shuffle=True,                    num_workers=4, worker_init_fn=worker_init_fn)

## collate_fnを自作する ― 可変サイズ・メタデータ・壊れた症例に対応する

In [ ]:
from torch.utils.data import default_collatedef safe_collate(batch):    kept, failed = [], []    for b in batch:        if isinstance(b, dict) and b.get("error"):   # Dataset側が返した失敗レコード            failed.append((b["case_id"], b["error"]))            continue        kept.append(b)    # 成功分と失敗分を「組」にして返す。collate_fn は num_workers>0 のとき worker プロセスの中で    # 動くので、グローバルのリストに追記しても親プロセスには届かない。戻り値で持ち帰る。    return {"data": default_collate(kept) if kept else None, "failed": failed}# Dataset側: __getitem__ を try/except で包み、失敗時は {"case_id":..., "error": str(e)} を返す# 学習ループ側（親プロセス）:#   FAILED = []                               # 落とした症例を必ず残す。件数と理由が分からない除外は成績を歪める#   for batch in loader:#       FAILED.extend(batch["failed"])        # worker から戻ってきた失敗分を回収する#       if batch["data"] is None: continue    # 全滅バッチはスキップ（件数は FAILED に残る）#       imgs, labels = batch["data"]# 学習の最後に: 全対象 / 成功 / 失敗 を突き合わせ、FAILED を必ず出力する。# 評価では、失敗件数を分母から黙って外さず「評価不能」として報告するloader = DataLoader(ds, batch_size=16, collate_fn=safe_collate, num_workers=4)

In [ ]:
import torch, torch.nn.functional as Fdef pad_collate(batch):    imgs, labels = zip(*batch)                       # 各 img は (C, H, W)、大きさばらばら    H = max(i.shape[-2] for i in imgs)    W = max(i.shape[-1] for i in imgs)    padded = [F.pad(i, (0, W - i.shape[-1], 0, H - i.shape[-2])) for i in imgs]  # 右・下に0詰め    return torch.stack(padded), torch.tensor(labels)

In [ ]:
def meta_collate(batch):    imgs   = torch.stack([b["image"] for b in batch])    labels = torch.stack([b["label"] for b in batch])    meta   = [b["meta"] for b in batch]              # {"case_id":..., "spacing":...} のリスト    return {"image": imgs, "label": labels, "meta": meta}

## 23.4　拡張は、学習と検証で分ける

In [ ]:
train_ds = VolumeDataset(train_df, transform=train_tf)   # 拡張ありval_ds   = VolumeDataset(val_df,   transform=val_tf)     # 拡張なし